In [5]:
import pandas as pd # import pandas library — the standard Python tool for working with tabular data

In [10]:
# This reads the key-value pairs from the .env file and loads them into the current process's environment variables. 
# It returns True if it successfully found and loaded a .env file, or False if it couldn't find one
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from openai import OpenAI # imports the OpenAI client class from the openai Python package.
openai_client = OpenAI() # creates an instance of that client, which handles authentication and communication with OpenAI's API.

# Generating Ground Truth Data

In [2]:
# We need questions with known relevant documents.
# We generate these with an LLM, asking it to create questions for each exercise:

# Loading the documents

In [12]:
df = pd.read_csv('../data/data.csv')
documents = df.to_dict(orient='records')

In [ ]:
# We'll generate questions for the fitness dataset. 
# The full Fitness dataset contains documents that cover different muscle groups, equipment types, and difficulty levels.

# prompt template for question generation

In [13]:
prompt_template = """
You are a user of a fitness assistant app. Based on the exercise record below,
generate 5 questions a user might ask that this record would be the best answer to.
The questions should be complete and not too short — imagine a real user typing them
without seeing the record itself.

Exercise record:
exercise_name: {exercise_name}
type_of_activity: {type_of_activity}
type_of_equipment: {type_of_equipment}
body_part: {body_part}
type: {type}
muscle_groups_activated: {muscle_groups_activated}
instructions: {instructions}

Provide the output in parsable JSON, as a list of 5 strings, with no other text:

["question1", "question2", ..., "question5"]
""".strip()

# function to call the LLM for one record

In [22]:
documents[0]

{'id': 'push-up-001',
 'exercise_name': 'Push-Up',
 'type_of_activity': 'Strength',
 'type_of_equipment': 'None (bodyweight)',
 'body_part': 'Chest',
 'type': 'Compound',
 'muscle_groups_activated': 'Chest, Triceps, Shoulders, Core',
 'instructions': 'Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.'}

In [23]:
prompt = prompt_template.format(**documents[0])

In [24]:
prompt

'You are a user of a fitness assistant app. Based on the exercise record below,\ngenerate 5 questions a user might ask that this record would be the best answer to.\nThe questions should be complete and not too short — imagine a real user typing them\nwithout seeing the record itself.\n\nExercise record:\nexercise_name: Push-Up\ntype_of_activity: Strength\ntype_of_equipment: None (bodyweight)\nbody_part: Chest\ntype: Compound\nmuscle_groups_activated: Chest, Triceps, Shoulders, Core\ninstructions: Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.\n\nProvide the output in parsable JSON, as a list of 5 strings, with no other text:\n\n["question1", "question2", ..., "question5"]'

In [27]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [28]:
questions = llm(prompt)

In [29]:
questions

'["What are some effective bodyweight exercises that target the chest and triceps?", "Can you explain the proper form for doing push-ups to avoid injury?", "What muscles are worked when performing a push-up, and how can I ensure I\'m engaging them properly?", "Are push-ups a good exercise for building upper body strength without any equipment?", "What instructions should I follow to perform a push-up correctly and maximize its benefits?"]'

# Generating Ground Truth for All Documents

In [30]:
import json
from tqdm.auto import tqdm

results = {}

for doc in tqdm(documents):
    doc_id = doc['id']
    if doc_id in results:
        continue  # skip if already processed (lets you safely re-run after a failure)
    
    prompt = prompt_template.format(**doc)
    questions_raw = llm(prompt)
    results[doc_id] = questions_raw

  0%|          | 0/50 [00:00<?, ?it/s]

In [31]:
# --- Parse the raw JSON strings into actual lists ---
parsed_results = {}

for doc_id, questions_raw in results.items():
    try:
        parsed_results[doc_id] = json.loads(questions_raw)
    except json.JSONDecodeError:
        print(f"Failed to parse for {doc_id}: {questions_raw[:200]}")


# --- Flatten into (id, question) pairs ---
final_results = []

for doc_id, questions in parsed_results.items():
    for q in questions:
        final_results.append((doc_id, q))

In [32]:
print(len(final_results))       # should be ~250 (50 exercises × 5 questions)
print(final_results[:3])        # should show clean (id, question) tuples

250
[('push-up-001', 'What are some effective bodyweight exercises I can do to strengthen my chest?'), ('push-up-001', 'Can you explain how to properly perform a push-up in detail?'), ('push-up-001', 'What muscle groups are activated when I do push-ups?')]


In [33]:
final_results[0]

('push-up-001',
 'What are some effective bodyweight exercises I can do to strengthen my chest?')

In [34]:
df_ground_truth = pd.DataFrame(final_results, columns=['id', 'question'])
df_ground_truth.to_csv('../data/ground-truth-data.csv', index=False)

In [35]:
!head ../data/ground-truth-data.csv

id,question
push-up-001,What are some effective bodyweight exercises I can do to strengthen my chest?
push-up-001,Can you explain how to properly perform a push-up in detail?
push-up-001,What muscle groups are activated when I do push-ups?
push-up-001,"Do push-ups require any equipment, and how do I set up for them?"
push-up-001,What is the correct form for doing a push-up to avoid injury?
bodyweight-squat-002,What is a good bodyweight exercise for strengthening my legs without any equipment?
bodyweight-squat-002,Can you provide instructions on how to perform a bodyweight squat effectively?
bodyweight-squat-002,What muscle groups are targeted when doing bodyweight squats?
bodyweight-squat-002,What are the proper form and technique for performing bodyweight squats?


In [36]:
df_ground_truth

,id,question
0,push-up-001,What are some effective bodyweight exercises I...
1,push-up-001,Can you explain how to properly perform a push...
2,push-up-001,What muscle groups are activated when I do pus...
3,push-up-001,"Do push-ups require any equipment, and how do ..."
4,push-up-001,What is the correct form for doing a push-up t...
...,...,...
245,donkey-kick-050,What is the Donkey Kick exercise and what musc...
246,donkey-kick-050,Can you provide instructions on how to properl...
247,donkey-kick-050,Is the Donkey Kick considered a strength exerc...
248,donkey-kick-050,What body parts are primarily engaged when doi...


In [37]:
df_ground_truth.head() # see top 5

,id,question
0,push-up-001,What are some effective bodyweight exercises I...
1,push-up-001,Can you explain how to properly perform a push...
2,push-up-001,What muscle groups are activated when I do pus...
3,push-up-001,"Do push-ups require any equipment, and how do ..."
4,push-up-001,What is the correct form for doing a push-up t...


In [39]:
len(df_ground_truth)

250

In [40]:
ground_truth = df_ground_truth.to_dict(orient='records') #easier to deal with dictionaries

In [41]:
ground_truth [0]

{'id': 'push-up-001',
 'question': 'What are some effective bodyweight exercises I can do to strengthen my chest?'}

# Evaluating retrieval quality
